In [1]:
# ============================================================
# INDIAN QUANT PORTFOLIO & RISK ENGINE
# STEP 4: WALK-FORWARD BACKTESTING
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.optimize import minimize


# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "data/raw"
OUTPUT_DIR = "data/processed/backtest"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRADING_DAYS = 252

# One year of historical data used for optimization
LOOKBACK = 252

# Rebalance every 3 months
REBALANCE_MONTHS = 3

# Approximate total transaction cost
# Change this after deciding your final assumption.
TRANSACTION_COST = 0.001

# Portfolio constraints
MAX_WEIGHT = 0.15
MIN_WEIGHT = 0.00

# Use zero initially; can later be replaced
# with an Indian risk-free rate.
RISK_FREE_RATE = 0.0


# ============================================================
# LOAD DATA
# ============================================================

returns = pd.read_csv(
    os.path.join(DATA_DIR, "daily_returns.csv"),
    index_col="Date",
    parse_dates=True
)

returns = returns.sort_index()

returns = returns.replace(
    [np.inf, -np.inf],
    np.nan
)

# Keep stocks with sufficiently long history
min_obs = int(len(returns) * 0.90)

valid_stocks = [
    c for c in returns.columns
    if returns[c].count() >= min_obs
]

returns = returns[valid_stocks]

print("=" * 70)
print("WALK-FORWARD BACKTEST")
print("=" * 70)

print(f"\nStocks: {len(valid_stocks)}")
print(f"Data: {returns.index.min().date()} → "
      f"{returns.index.max().date()}")


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def portfolio_return(weights, expected_returns):
    return np.dot(weights, expected_returns)


def portfolio_volatility(weights, covariance):
    return np.sqrt(
        weights.T @ covariance @ weights
    )


def portfolio_sharpe(weights, expected_returns, covariance):
    vol = portfolio_volatility(
        weights,
        covariance
    )

    if vol == 0:
        return 0

    return (
        portfolio_return(
            weights,
            expected_returns
        ) - RISK_FREE_RATE
    ) / vol


def risk_contribution(weights, covariance):

    portfolio_vol = portfolio_volatility(
        weights,
        covariance
    )

    marginal = covariance @ weights

    contribution = (
        weights * marginal
    )

    return contribution / portfolio_vol


# ============================================================
# PORTFOLIO OPTIMIZERS
# ============================================================

def equal_weight(returns_window):

    n = returns_window.shape[1]

    return np.ones(n) / n


def minimum_variance(returns_window):

    covariance = (
        returns_window.cov().values
        * TRADING_DAYS
    )

    n = len(returns_window.columns)

    initial = np.ones(n) / n

    bounds = [
        (MIN_WEIGHT, MAX_WEIGHT)
        for _ in range(n)
    ]

    constraints = {
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1
    }

    result = minimize(
        lambda w:
        w.T @ covariance @ w,
        initial,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": 1000,
            "ftol": 1e-9
        }
    )

    return result.x


def maximum_sharpe(returns_window):

    expected = (
        returns_window.mean()
        * TRADING_DAYS
    ).values

    covariance = (
        returns_window.cov().values
        * TRADING_DAYS
    )

    n = len(returns_window.columns)

    initial = np.ones(n) / n

    bounds = [
        (MIN_WEIGHT, MAX_WEIGHT)
        for _ in range(n)
    ]

    constraints = {
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1
    }

    result = minimize(
        lambda w:
        -portfolio_sharpe(
            w,
            expected,
            covariance
        ),
        initial,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": 2000,
            "ftol": 1e-9
        }
    )

    return result.x


def risk_parity(returns_window):

    covariance = (
        returns_window.cov().values
        * TRADING_DAYS
    )

    n = len(returns_window.columns)

    initial = np.ones(n) / n

    bounds = [
        (MIN_WEIGHT, MAX_WEIGHT)
        for _ in range(n)
    ]

    constraints = {
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1
    }

    def objective(w):

        rc = risk_contribution(
            w,
            covariance
        )

        target = 1 / n

        return np.sum(
            (rc - target) ** 2
        )

    result = minimize(
        objective,
        initial,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": 2000,
            "ftol": 1e-10
        }
    )

    return result.x


# ============================================================
# STRATEGIES
# ============================================================

strategies = {
    "Equal_Weight": equal_weight,
    "Minimum_Variance": minimum_variance,
    "Maximum_Sharpe": maximum_sharpe,
    "Risk_Parity": risk_parity
}


# ============================================================
# REBALANCING DATES
# ============================================================

dates = returns.index

first_date = dates[LOOKBACK]

rebalance_dates = pd.date_range(
    start=first_date,
    end=dates[-1],
    freq=f"{REBALANCE_MONTHS}MS"
)

rebalance_dates = [
    dates[
        dates.searchsorted(date)
    ]
    for date in rebalance_dates
]

rebalance_dates = sorted(
    set(rebalance_dates)
)


# ============================================================
# BACKTEST ENGINE
# ============================================================

results = {}

weights_history = {}

turnover_history = {}


for strategy_name, optimizer in strategies.items():

    print(
        f"\nRunning: {strategy_name}"
    )

    portfolio_returns = pd.Series(
        index=dates,
        dtype=float
    )

    strategy_weights = {}

    strategy_turnover = {}

    previous_weights = None

    for i, rebalance_date in enumerate(
        rebalance_dates
    ):

        # ----------------------------------------------------
        # TRAINING WINDOW
        # ----------------------------------------------------

        current_position = dates.get_loc(
            rebalance_date
        )

        if current_position < LOOKBACK:
            continue

        train_start = (
            current_position - LOOKBACK
        )

        train_end = current_position

        training_data = returns.iloc[
            train_start:train_end
        ].copy()

        # Remove rows with missing values
        training_data = training_data.dropna()

        if len(training_data) < 200:
            continue

        # ----------------------------------------------------
        # OPTIMIZE
        # ----------------------------------------------------

        weights = optimizer(
            training_data
        )

        weights = np.asarray(
            weights,
            dtype=float
        )

        # Numerical cleanup
        weights[
            np.abs(weights) < 1e-8
        ] = 0

        weights = (
            weights / weights.sum()
        )

        # ----------------------------------------------------
        # TURNOVER
        # ----------------------------------------------------

        if previous_weights is None:

            turnover = np.sum(
                np.abs(weights)
            )

        else:

            turnover = np.sum(
                np.abs(
                    weights
                    - previous_weights
                )
            )

        strategy_turnover[
            rebalance_date
        ] = turnover

        strategy_weights[
            rebalance_date
        ] = weights

        # ----------------------------------------------------
        # TRANSACTION COST
        # ----------------------------------------------------

        transaction_cost = (
            turnover
            * TRANSACTION_COST
        )

        # ----------------------------------------------------
        # TESTING WINDOW
        # ----------------------------------------------------

        if i + 1 < len(rebalance_dates):

            next_rebalance = (
                rebalance_dates[i + 1]
            )

            test_data = returns.loc[
                rebalance_date:
                next_rebalance
            ].iloc[:-1]

        else:

            test_data = returns.loc[
                rebalance_date:
            ]

        test_data = test_data.dropna(
            how="all"
        )

        # ----------------------------------------------------
        # PORTFOLIO RETURNS
        # ----------------------------------------------------

        weighted_returns = (
            test_data
            .fillna(0)
            .dot(weights)
        )

        # Apply transaction cost at
        # beginning of testing period
        if len(weighted_returns) > 0:

            weighted_returns.iloc[0] -= (
                transaction_cost
            )

        portfolio_returns.loc[
            weighted_returns.index
        ] = weighted_returns

        previous_weights = weights

    # Remove empty observations
    portfolio_returns = (
        portfolio_returns
        .dropna()
    )

    results[
        strategy_name
    ] = portfolio_returns

    weights_history[
        strategy_name
    ] = strategy_weights

    turnover_history[
        strategy_name
    ] = strategy_turnover


# ============================================================
# COMBINE RETURNS
# ============================================================

backtest_returns = pd.DataFrame(
    results
)


# ============================================================
# CUMULATIVE PERFORMANCE
# ============================================================

cumulative_returns = (
    1 + backtest_returns
).cumprod()


# ============================================================
# PERFORMANCE METRICS
# ============================================================

def calculate_metrics(series):

    series = series.dropna()

    if len(series) == 0:
        return {}

    total_return = (
        1 + series
    ).prod() - 1

    years = (
        len(series) / TRADING_DAYS
    )

    cagr = (
        (1 + total_return)
        ** (1 / years)
        - 1
    )

    volatility = (
        series.std()
        * np.sqrt(TRADING_DAYS)
    )

    sharpe = (
        (series.mean() * TRADING_DAYS)
        / volatility
        if volatility > 0
        else np.nan
    )

    downside = (
        series.clip(upper=0)
        .std()
        * np.sqrt(TRADING_DAYS)
    )

    sortino = (
        (series.mean() * TRADING_DAYS)
        / downside
        if downside > 0
        else np.nan
    )

    cumulative = (
        1 + series
    ).cumprod()

    running_max = cumulative.cummax()

    drawdown = (
        cumulative / running_max
    ) - 1

    max_drawdown = drawdown.min()

    calmar = (
        cagr / abs(max_drawdown)
        if max_drawdown < 0
        else np.nan
    )

    var_95 = series.quantile(
        0.05
    )

    cvar_95 = series[
        series <= var_95
    ].mean()

    return {

        "Total_Return": total_return,

        "CAGR": cagr,

        "Annualized_Volatility":
            volatility,

        "Sharpe_Ratio":
            sharpe,

        "Sortino_Ratio":
            sortino,

        "Maximum_Drawdown":
            max_drawdown,

        "Calmar_Ratio":
            calmar,

        "VaR_95":
            var_95,

        "CVaR_95":
            cvar_95,

        "Best_Day":
            series.max(),

        "Worst_Day":
            series.min(),

        "Positive_Days_%":
            (series > 0).mean(),

        "Observations":
            len(series)
    }


performance = pd.DataFrame({

    strategy:
        calculate_metrics(
            backtest_returns[strategy]
        )

    for strategy in backtest_returns.columns

}).T


# ============================================================
# TURNOVER SUMMARY
# ============================================================

turnover_summary = {}

for strategy, values in turnover_history.items():

    turnover_values = list(
        values.values()
    )

    turnover_summary[strategy] = {

        "Average_Turnover":
            np.mean(turnover_values),

        "Median_Turnover":
            np.median(turnover_values),

        "Total_Turnover":
            np.sum(turnover_values),

        "Number_of_Rebalances":
            len(turnover_values)
    }


turnover_summary = pd.DataFrame(
    turnover_summary
).T


# ============================================================
# SAVE RESULTS
# ============================================================

backtest_returns.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "backtest_daily_returns.csv"
    )
)

cumulative_returns.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "backtest_cumulative_returns.csv"
    )
)

performance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "backtest_performance.csv"
    )
)

turnover_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "turnover_summary.csv"
    )
)


# ============================================================
# SAVE WEIGHT HISTORY
# ============================================================

for strategy, history in weights_history.items():

    if history:

        weight_df = pd.DataFrame(
            history
        ).T

        weight_df.columns = valid_stocks

        weight_df.to_csv(
            os.path.join(
                OUTPUT_DIR,
                f"{strategy}_weights_history.csv"
            )
        )


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BACKTEST PERFORMANCE")
print("=" * 70)

display_columns = [
    "CAGR",
    "Annualized_Volatility",
    "Sharpe_Ratio",
    "Sortino_Ratio",
    "Maximum_Drawdown",
    "Calmar_Ratio",
    "VaR_95",
    "CVaR_95"
]

print(
    performance[
        display_columns
    ]
    .sort_values(
        "Sharpe_Ratio",
        ascending=False
    )
    .round(4)
)


print("\n" + "=" * 70)
print("TURNOVER")
print("=" * 70)

print(
    turnover_summary.round(4)
)


print("\n" + "=" * 70)
print("BACKTEST COMPLETE")
print("=" * 70)

print(
    f"\nResults saved to: {OUTPUT_DIR}"
)

WALK-FORWARD BACKTEST

Stocks: 15
Data: 2015-01-01 → 2026-08-19

Running: Equal_Weight

Running: Minimum_Variance

Running: Maximum_Sharpe

Running: Risk_Parity

BACKTEST PERFORMANCE
                    CAGR  Annualized_Volatility  Sharpe_Ratio  Sortino_Ratio  \
Maximum_Sharpe    0.1835                 0.1594        1.1372         1.8239   
Equal_Weight      0.1741                 0.1533        1.1241         1.7934   
Minimum_Variance  0.1437                 0.1383        1.0406         1.6932   
Risk_Parity       0.1850                 0.1956        0.9664         1.5444   

                  Maximum_Drawdown  Calmar_Ratio  VaR_95  CVaR_95  
Maximum_Sharpe             -0.2995        0.6125 -0.0150  -0.0227  
Equal_Weight               -0.3238        0.5377 -0.0143  -0.0219  
Minimum_Variance           -0.2879        0.4991 -0.0123  -0.0190  
Risk_Parity                -0.3594        0.5148 -0.0199  -0.0283  

TURNOVER
                  Average_Turnover  Median_Turnover  Total_Turnove